## Example 6c: Antithetic Integral Controller — comparison with the LASSO of Zhang et al.

This notebook compares the Bayesian results of `example6_aic_Tsweep.ipynb` with the $\ell_1$-penalised likelihood method
of Zhang et al. (2019), sparse-learning-CRN (https://github.com/zwpku/sparse-learning-CRN), on the same prefix
trajectories of the ergodic antithetic integral controller ($T \in \{100, 500, 1000, 5000\}$, $\lambda = 0.01$).
**Their C++ code is not modified.** The helper functions are in `src/sparse_learning_compare.py`.

### Two configurations of their method

| | Black box (**LASSO-BB** in the paper) | Mass action (**LASSO-MA** in the paper) |
|---|---|---|
| Basis per channel | all 10 polynomials of order $\le 2$ in $(X, Z_1, Z_2)$ (their default) | only the propensities of that channel's candidate reactions (the Bayesian candidate set) |
| Quadratic basis (`xx_basis_flag`) | $x^2$ (1, their default) | $x(x-1)$ (0, mass action) |
| Reported as | polynomial propensity per channel | rate constant per candidate reaction ($k = 2c$ for a $2X$ reactant) |
| Run folder | `sparse_runs/example6_T{T}_lam0p01_blackbox` | `sparse_runs/example6_T{T}_lam0p01_massaction` |

Both use their FISTA solver with a longer stopping rule than their defaults (`tot_step = 300000`,
`max_step_since_prev_min_cost = 100000`, `cost_stop_tol = 1e-12`), because the defaults can stop before convergence.

**Note on $\lambda$.** Their objective is the log-likelihood **divided by the observation time** plus
$\lambda \sum_j |c_j|$. With $\lambda$ fixed, the penalty therefore keeps the same weight relative to the data as $T$
grows: the shrinkage does not fade, and the estimates do not converge to the true rates (for example, rates of
correlated candidate reactions keep sharing weight). A consistent LASSO would need $\lambda \to 0$ as $T \to \infty$.
We keep $\lambda = 0.01$ at every $T$, the value used throughout the paper.

### Which fits are computed by their program

All fits, at every $T$ and in both configurations, are computed with their program (`PROGRAM_T`), with the long
stopping rule above. Their solver evaluates the objective over every recorded event at every iteration and needs
$3.6\times10^4$ to $3\times10^5$ iterations per channel, so the long fits take hours. Measured on a MacBook Air
(sum over the five channels, which were run in parallel with `scripts/run_lasso_overnight.sh`):

| $T$ | LASSO-MA | LASSO-BB |
|---|---|---|
| 1000 | 0.9 h | 3.0 h |
| 5000 | 5.6 h | 14.5 h |

Earlier versions of this notebook used our independent solver of the same objective at $T = 1000, 5000$; their program
now reproduces those solutions (objective values within $2\times10^{-8}$, coefficients within $2\times10^{-3}$).
If a $T$ is removed from `PROGRAM_T`, the notebook falls back to the independent solver for it and marks it with $^\ast$.

### Independent check
For every run, `check_run` re-implements their objective, evaluates it at their coefficients, minimises it with a
different optimiser (L-BFGS-B), and reports the gap. A gap near zero means their result is the LASSO solution.

### Modes
* `RUN_LASSO = False` (default): post-processing only. Reads the run folders and the Bayesian summaries in
  `results/example6_T{T}/`, runs the independent check, and writes the tables (a few minutes, mostly the check).
* `RUN_LASSO = True`: re-runs their programs (`prepare` at every $T$, `sparse_learning` at $T$ in `PROGRAM_T`), one
  channel after another: about a day in total, mostly $T = 5000$. To run the channels in parallel use
  `scripts/run_lasso_overnight.sh` instead (see `examples/README.md`).

### Outputs
* `results/example6_lasso/`: comparison tables as CSV.
* `writing/table_aic_lasso_comparison.tex`: active rates, Bayesian vs. mass-action LASSO, versus $T$.
* `writing/table_aic_lasso_blackbox.tex`: black-box polynomial propensities versus $T$.

Run `example6_aic.ipynb` (trajectory) and `example6_aic_Tsweep.ipynb` (prefixes and Bayesian results) first.


### Settings

In [1]:
import os, sys, json
import numpy as np
import pandas as pd

REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
from src.parsing import load_reaction_network, generate_reactions
from src.sparse_learning_compare import (prepare_run, run_program, restrict_to_candidates, read_channel_info,
                                         read_basis_info, channel_basis, read_omega, cfg_value, basis_label,
                                         reactant_key, rate_factor, termination_summary, check_run, DEFAULT_STOP)

# False: post-processing only.  True: re-run their programs (several hours).
RUN_LASSO = False
OVERWRITE = False                      # with RUN_LASSO = True: allow replacing existing run folders

SPARSE_LEARNING_SRC = os.environ.get(
    "SPARSE_LEARNING_SRC",
    "/Users/suzannesindi/Dropbox/WritingProjects/2026_06_08_CRNWork/sparse-learning-CRN/src")
PREPARE_EXE = os.path.join(SPARSE_LEARNING_SRC, "prepare")
SPARSE_EXE  = os.path.join(SPARSE_LEARNING_SRC, "sparse_learning")

T_LIST    = [100, 500, 1000, 5000]
PROGRAM_T = [100, 500, 1000, 5000]     # fits run with their program; at other T our solver of their objective is used
LAMBDA             = 0.01
STOP               = DEFAULT_STOP

NETWORK_FILE = os.path.join(REPO, "data", "example6_network.json")
def traj_file(T):
    return os.path.join(REPO, "data", "example6_trajectory.json" if T == 5000 else f"example6_T{T}_trajectory.json")
def run_dir(T, config):
    return os.path.join(REPO, "sparse_runs", f"example6_T{T}_lam{str(LAMBDA).replace('.', 'p')}_{config}")
def bayes_file(T):
    return os.path.join(REPO, "results", f"example6_T{T}", "mcmc_summary.xlsx")

OUT_DIR = os.path.join(REPO, "results", "example6_lasso")
os.makedirs(OUT_DIR, exist_ok=True)

for T in T_LIST:
    for p in [traj_file(T), bayes_file(T)]:
        print(("ok      " if os.path.exists(p) else "MISSING ") + os.path.relpath(p, REPO))
if RUN_LASSO:
    for p in [PREPARE_EXE, SPARSE_EXE]:
        print(("ok      " if os.path.exists(p) else "MISSING ") + p)
print("Mode:", "re-run their programs" if RUN_LASSO else "post-processing only")


ok      data/example6_T100_trajectory.json
ok      results/example6_T100/mcmc_summary.xlsx
ok      data/example6_T500_trajectory.json
ok      results/example6_T500/mcmc_summary.xlsx
ok      data/example6_T1000_trajectory.json
ok      results/example6_T1000/mcmc_summary.xlsx
ok      data/example6_trajectory.json
ok      results/example6_T5000/mcmc_summary.xlsx
Mode: post-processing only


### Network and candidate reactions

In [2]:
import io, contextlib
with contextlib.redirect_stdout(io.StringIO()):
    (_, _, _, _, trueTheta, _, _, reactant_matrix, unique_changes, compatible_reactions,
     species) = load_reaction_network(NETWORK_FILE)
complexes = np.array([[0, 1, 0, 0, 2, 0, 0, 1, 1, 0],
                      [0, 0, 1, 0, 0, 2, 0, 1, 0, 1],
                      [0, 0, 0, 1, 0, 0, 2, 0, 1, 1]])
with contextlib.redirect_stdout(io.StringIO()):
    reaction_names = [r.rstrip(":") for r in generate_reactions(complexes, species)[3]]
CHANNELS = [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, 0, 1), (0, -1, -1)]   # observed channels, in table order
for ch in CHANNELS:
    print(ch, [f"{reaction_names[i]} ({trueTheta[i]:g})" for i in compatible_reactions[ch]])


(1, 0, 0) ['Empty_to_X (2)', 'X_to_2X (0)', 'Z1_to_X+Z1 (1)', 'Z2_to_X+Z2 (0)']
(-1, 0, 0) ['X_to_Empty (1)', '2X_to_X (0)', 'X+Z1_to_Z1 (0)', 'X+Z2_to_Z2 (0)']
(0, 1, 0) ['Empty_to_Z1 (6)', 'X_to_X+Z1 (0)', 'Z1_to_2Z1 (0)', 'Z2_to_Z1+Z2 (0)']
(0, 0, 1) ['Empty_to_Z2 (0)', 'X_to_X+Z2 (2)', 'Z1_to_Z1+Z2 (0)', 'Z2_to_2Z2 (0)']
(0, -1, -1) ['Z1+Z2_to_Empty (1)']


### Running their programs (only when `RUN_LASSO = True`)

For each $T$: export the trajectory and write the configuration; run `prepare`; for the mass-action configuration, cut
each channel's basis file down to its candidate reactions; run `sparse_learning` if $T$ is in `PROGRAM_T` (otherwise
only `prepare` is run: it defines the channels and the basis for our solver).

In [3]:
KEEP = {}
for T in T_LIST:
    for config, xx in [("massaction", 0), ("blackbox", 1)]:
        d = run_dir(T, config)
        if RUN_LASSO:
            prepare_run(d, traj_file(T), T, LAMBDA, xx_flag=xx, stop=STOP, overwrite=OVERWRITE)
            run_program(PREPARE_EXE, d)
        if config == "massaction":
            KEEP[T] = restrict_to_candidates(d, reactant_matrix, compatible_reactions, write=RUN_LASSO)
        if RUN_LASSO and T in PROGRAM_T:
            print(f"T = {T}, {config}: running sparse_learning ...", flush=True)
            run_program(SPARSE_EXE, d)
        assert int(cfg_value(d, "xx_basis_flag")) == xx and cfg_value(d, "regular_lambda") == LAMBDA
        print(f"T = {T:5d}  {config:10s}  {os.path.relpath(d, REPO)}")


T =   100  massaction  sparse_runs/example6_T100_lam0p01_massaction
T =   100  blackbox    sparse_runs/example6_T100_lam0p01_blackbox
T =   500  massaction  sparse_runs/example6_T500_lam0p01_massaction
T =   500  blackbox    sparse_runs/example6_T500_lam0p01_blackbox
T =  1000  massaction  sparse_runs/example6_T1000_lam0p01_massaction
T =  1000  blackbox    sparse_runs/example6_T1000_lam0p01_blackbox
T =  5000  massaction  sparse_runs/example6_T5000_lam0p01_massaction
T =  5000  blackbox    sparse_runs/example6_T5000_lam0p01_blackbox


### Convergence: their stopping reason and the independent check

`gap` = objective at their coefficients minus objective at our optimum (`NaN` where their program was not run).

Our optimum is saved in each run folder (`output/independent_optimum.json`) when the fits are run, and read back in
post-processing mode, so the tables do not depend on the installed scipy version. The optimisation is still repeated
here: `re-solve diff` is the objective at the fresh optimum minus the objective at the saved one. Values of order
$10^{-6}$ or smaller mean the two agree; on very flat objectives (black box) the coefficients of a fresh solve can
differ in the third decimal while the objective agrees.

In [4]:
pd.set_option("display.float_format", lambda v: f"{v:.6g}")
pd.set_option("display.width", 200)
checks = {}
for T in T_LIST:
    for config in ["massaction", "blackbox"]:
        d = run_dir(T, config)
        checks[(T, config)] = check_run(d, species, save=RUN_LASSO)
        term = termination_summary(d)
        print(f"\n===== T = {T}, {config}")
        if term is not None:
            print(term.to_string(index=False))
        print(checks[(T, config)][["channel", "their cost (log)", "f(their coeffs)", "f(our optimum)", "gap",
                                   "re-solve diff"]].to_string(index=False))

gaps = pd.DataFrame([{"T": T, "config": c, "largest gap": df["gap"].max()} for (T, c), df in checks.items()])
print("\nLargest gap per run (NaN: their program not run, our solver used):")
print(gaps.pivot(index="T", columns="config", values="largest gap").to_string())
resolve = pd.DataFrame([{"T": T, "config": c, "re-solve": df["re-solve diff"].abs().max()} for (T, c), df in checks.items()])
print("\nLargest |re-solve diff| per run:")
print(resolve.pivot(index="T", columns="config", values="re-solve").to_string())



===== T = 100, massaction
    channel                                                         reason  steps
 (-1, 0, 0) Changes of the costs in the last 20 steps are under threshold. 161300
(0, -1, -1) Changes of the costs in the last 20 steps are under threshold.  42372
  (0, 0, 1) Changes of the costs in the last 20 steps are under threshold. 126628
  (0, 1, 0)           Minimal cost hasn't been updated after 100000 steps. 248669
  (1, 0, 0) Changes of the costs in the last 20 steps are under threshold. 205155
    channel  their cost (log)  f(their coeffs)  f(our optimum)         gap  re-solve diff
 (-1, 0, 0)          -1.29823         -1.29823        -1.29823 1.11022e-15    1.11022e-15
(0, -1, -1)          -11.0046         -11.0046        -11.0046           0              0
  (0, 0, 1)          -6.46039         -6.46039        -6.46039 8.44658e-13    1.77636e-15
  (0, 1, 0)           -4.7032         -4.70319        -4.70319 2.08029e-11   -2.22045e-14
  (1, 0, 0)         -0.911009  


===== T = 100, blackbox
    channel                                                         reason  steps
 (-1, 0, 0) Changes of the costs in the last 20 steps are under threshold. 137284
(0, -1, -1) Changes of the costs in the last 20 steps are under threshold.  55666
  (0, 0, 1) Changes of the costs in the last 20 steps are under threshold. 174010
  (0, 1, 0) Changes of the costs in the last 20 steps are under threshold. 165485
  (1, 0, 0)                               Maximal steps (300000) achieved. 300000
    channel  their cost (log)  f(their coeffs)  f(our optimum)         gap  re-solve diff
 (-1, 0, 0)          -1.32055         -1.32055        -1.32055 5.45656e-10   -4.55191e-14
(0, -1, -1)           -11.043          -11.043         -11.043 1.62947e-09   -7.88489e-11
  (0, 0, 1)          -6.46448         -6.46448        -6.46448 1.09501e-09    2.85461e-12
  (0, 1, 0)          -4.73483         -4.73483        -4.73483 7.13383e-09    3.19744e-13
  (1, 0, 0)         -0.968419    


===== T = 500, massaction
    channel                                                         reason  steps
 (-1, 0, 0) Changes of the costs in the last 20 steps are under threshold. 178323
(0, -1, -1) Changes of the costs in the last 20 steps are under threshold.  45175
  (0, 0, 1) Changes of the costs in the last 20 steps are under threshold. 127273
  (0, 1, 0) Changes of the costs in the last 20 steps are under threshold. 283183
  (1, 0, 0) Changes of the costs in the last 20 steps are under threshold. 283787
    channel  their cost (log)  f(their coeffs)  f(our optimum)         gap  re-solve diff
 (-1, 0, 0)           -1.1053          -1.1053         -1.1053 2.22045e-15    1.77636e-15
(0, -1, -1)          -10.7476         -10.7475        -10.7475 4.08562e-14    7.10543e-15
  (0, 0, 1)          -5.90038         -5.90038        -5.90038           0              0
  (0, 1, 0)          -4.61504         -4.61504        -4.61504 6.20659e-12   -3.57936e-13
  (1, 0, 0)         -0.667613  


===== T = 500, blackbox
    channel                                                         reason  steps
 (-1, 0, 0) Changes of the costs in the last 20 steps are under threshold. 212815
(0, -1, -1) Changes of the costs in the last 20 steps are under threshold.  57032
  (0, 0, 1) Changes of the costs in the last 20 steps are under threshold. 170812
  (0, 1, 0) Changes of the costs in the last 20 steps are under threshold. 189348
  (1, 0, 0) Changes of the costs in the last 20 steps are under threshold. 227892
    channel  their cost (log)  f(their coeffs)  f(our optimum)         gap  re-solve diff
 (-1, 0, 0)           -1.1181         -1.11811        -1.11811 2.30909e-10   -3.59712e-14
(0, -1, -1)          -10.7838         -10.7838        -10.7838 4.40465e-11    3.67706e-13
  (0, 0, 1)          -5.90128         -5.90128        -5.90128 2.00867e-09    1.25233e-13
  (0, 1, 0)          -4.62135         -4.62135        -4.62135 9.31325e-09    3.58824e-13
  (1, 0, 0)         -0.675019    


===== T = 1000, massaction
    channel  their cost (log)  f(their coeffs)  f(our optimum)          gap  re-solve diff
 (-1, 0, 0)               NaN         -1.15149        -1.15149  7.54952e-15   -4.21885e-15
(0, -1, -1)               NaN         -10.4977        -10.4977 -3.55271e-15   -7.10543e-15
  (0, 0, 1)               NaN         -5.83241        -5.83241 -1.76499e-11   -1.76499e-11
  (0, 1, 0)               NaN         -4.62943        -4.62943 -9.22995e-12   -1.01767e-11
  (1, 0, 0)               NaN        -0.785538       -0.785538  9.66593e-12   -1.64668e-12



===== T = 1000, blackbox
    channel  their cost (log)  f(their coeffs)  f(our optimum)          gap  re-solve diff
 (-1, 0, 0)               NaN         -1.15489        -1.15489  -2.1685e-09    -2.1685e-09
(0, -1, -1)               NaN         -10.5353        -10.5353 -1.16324e-06   -1.16467e-06
  (0, 0, 1)               NaN         -5.83428        -5.83428 -3.12081e-08   -3.12081e-08
  (0, 1, 0)               NaN         -4.63199        -4.63199  1.33425e-09   -2.71591e-10
  (1, 0, 0)               NaN         -0.78941        -0.78941  1.72787e-09   -1.06083e-11



===== T = 5000, massaction
    channel  their cost (log)  f(their coeffs)  f(our optimum)          gap  re-solve diff
 (-1, 0, 0)               NaN        -0.912387       -0.912387 -1.54432e-12   -1.54476e-12
(0, -1, -1)               NaN         -10.7729        -10.7729  5.32907e-14    3.19744e-14
  (0, 0, 1)               NaN         -5.97106        -5.97106 -2.76223e-12   -2.77112e-12
  (0, 1, 0)               NaN         -4.67698        -4.67698  2.21424e-10   -2.04281e-14
  (1, 0, 0)               NaN        -0.562448       -0.562448 -3.15015e-12   -3.57958e-12



===== T = 5000, blackbox
    channel  their cost (log)  f(their coeffs)  f(our optimum)          gap  re-solve diff
 (-1, 0, 0)               NaN        -0.913598       -0.913598 -2.41051e-08   -2.41051e-08
(0, -1, -1)               NaN         -10.8084        -10.8084 -5.20914e-07   -5.20965e-07
  (0, 0, 1)               NaN         -5.97149        -5.97149 -1.22536e-08    -2.0997e-08
  (0, 1, 0)               NaN         -4.67872        -4.67872   1.5866e-08    9.84102e-13
  (1, 0, 0)               NaN        -0.563943       -0.563943  -8.5927e-09   -8.60098e-09

Largest gap per run (NaN: their program not run, our solver used):
config    blackbox  massaction
T                             
100    7.13383e-09 2.08029e-11
500    9.31325e-09 6.20659e-12
1000   1.72787e-09 9.66593e-12
5000    1.5866e-08 2.21424e-10

Largest |re-solve diff| per run:
config    blackbox  massaction
T                             
100    7.88489e-11 2.22045e-14
500    3.67706e-13 3.57936e-13
1000   1.16467e-

### Estimates

In both configurations: their coefficients where their program was run, otherwise our solver's minimiser of their
objective (column `source`). Mass-action coefficients are converted to rate constants.

In [5]:
def bayes_table(T):
    if not os.path.exists(bayes_file(T)):
        return {}
    df = pd.read_excel(bayes_file(T))
    out = {}
    for _, r in df.iterrows():
        ch = unique_changes[int(r["Run_Index"])]
        out[compatible_reactions[ch][int(r["Param_Index"])]] = r
    return out

rows = []
def fitted_coefficients(T, config, ch_i):
    """Their coefficients if their program was run, otherwise our solver's optimum of their objective."""
    row = checks[(T, config)].iloc[ch_i]
    if row["their coeffs"] is not None:
        return row["their coeffs"], "their program"
    return row["our coeffs"], "our solver of their objective"

for T in T_LIST:
    d = run_dir(T, "massaction")
    changes, counts = read_channel_info(d)
    bayes = bayes_table(T)
    for ch_i, ch in enumerate(changes):
        coef, source = fitted_coefficients(T, "massaction", ch_i)
        for (b, gi), c in zip(KEEP[T][ch], coef):
            r = bayes.get(gi)
            rows.append({"T": T, "channel": ch, "events": counts[ch_i], "reaction": reaction_names[gi],
                         "true": trueTheta[gi], "lasso_ma": rate_factor(reactant_matrix[:, gi]) * c,
                         "bayes_mean": np.nan if r is None else r["Mean"],
                         "bayes_lo": np.nan if r is None else r["CI_lower"],
                         "bayes_hi": np.nan if r is None else r["CI_upper"],
                         "pr_off": np.nan if r is None else r["Prob_Off"], "source": source})
est = pd.DataFrame(rows)
est.to_csv(os.path.join(OUT_DIR, "massaction_vs_bayes.csv"), index=False)

act = est[est["true"] > 0]
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("Active reactions: mass-action LASSO estimate")
print(act.pivot_table(index="reaction", columns="T", values="lasso_ma").to_string())
print("\nActive reactions: Bayesian posterior mean")
print(act.pivot_table(index="reaction", columns="T", values="bayes_mean").to_string())
ina = est[est["true"] == 0]
print("\nInactive candidates: largest |LASSO estimate| and smallest Bayesian Pr(off), per T")
print(pd.DataFrame({"max |k| LASSO": ina.groupby("T")["lasso_ma"].apply(lambda s: s.abs().max()),
                    "# LASSO k < 0": ina.groupby("T")["lasso_ma"].apply(lambda s: int((s < 0).sum())),
                    "min Pr(off) Bayes": ina.groupby("T")["pr_off"].min()}).to_string())


Active reactions: mass-action LASSO estimate
T                100    500    1000   5000
reaction                                  
Empty_to_X     1.4725 2.0308 2.0335 1.9895
Empty_to_Z1    5.8094 5.9037 5.9115 5.7790
X_to_Empty     0.8282 0.9963 1.0252 0.9746
X_to_X+Z2      1.9099 2.0023 1.9512 2.0188
Z1+Z2_to_Empty 0.9814 1.0254 1.0146 1.0050
Z1_to_X+Z1     0.8694 0.9001 0.9435 0.9539

Active reactions: Bayesian posterior mean
T                100    500    1000   5000
reaction                                  
Empty_to_X     2.0224 2.1153 2.0962 2.0366
Empty_to_Z1    6.0041 5.9562 5.9645 5.9914
X_to_Empty     0.9766 1.0336 1.0378 1.0005
X_to_X+Z2      1.8963 1.9962 1.9515 1.9986
Z1+Z2_to_Empty 0.9866 1.0281 1.0168 1.0069
Z1_to_X+Z1     0.9647 0.9363 0.9428 0.9579

Inactive candidates: largest |LASSO estimate| and smallest Bayesian Pr(off), per T
      max |k| LASSO  # LASSO k < 0  min Pr(off) Bayes
T                                                    
100          0.2193             

In [6]:
def blackbox_poly(T, ch, thr=1e-3):
    """Black-box polynomial of channel ch at T: list of (coef, basis tuple), flag, and source of the estimate."""
    d = run_dir(T, "blackbox")
    changes, _ = read_channel_info(d)
    i = changes.index(ch)
    row = checks[(T, "blackbox")].iloc[i]
    use_theirs = row["their coeffs"] is not None
    coef = row["their coeffs"] if use_theirs else row["our coeffs"]
    basis = read_basis_info(d)
    allowed = {reactant_key(reactant_matrix[:, gi]) for gi in compatible_reactions[ch]}
    terms, bad = [], False
    for b, c in zip(channel_basis(d, i), coef):
        if abs(c) < thr:
            continue
        if tuple(sorted(basis[b])) not in allowed or c < 0:
            bad = True
        terms.append((c, basis[b]))
    return terms, bad, ("their program" if use_theirs else "our solver of their objective")

bb_rows = []
for T in T_LIST:
    for ch in CHANNELS:
        terms, bad, src = blackbox_poly(T, ch)
        bb_rows.append({"T": T, "channel": ch,
                        "polynomial": " ".join(f"{c:+.3f}·{basis_label(b, species, 1)}" for c, b in terms) + (" †" if bad else ""),
                        "source": src})
bb = pd.DataFrame(bb_rows)
bb.to_csv(os.path.join(OUT_DIR, "blackbox_polynomials.csv"), index=False)
pd.set_option("display.max_colwidth", 120)
print(bb.to_string(index=False))
print("\n† = contains a term that no candidate mass-action reaction of the channel can produce, or a negative term.")


   T     channel                                                                                                   polynomial        source
 100   (1, 0, 0)                      +0.903·X +1.386·Z1 +0.071·Z2 -0.076·X^2 +0.033·XZ1 -0.013·XZ2 -0.105·Z1^2 -0.001·Z2^2 † their program
 100  (-1, 0, 0)             -0.708·1 +1.084·X +0.416·Z1 -0.007·Z2 +0.004·X^2 -0.006·XZ1 -0.005·XZ2 -0.038·Z1^2 -0.007·Z1Z2 † their program
 100   (0, 1, 0) +4.159·1 +1.014·X -0.253·Z1 +0.094·Z2 -0.076·X^2 -0.011·XZ1 -0.022·XZ2 +0.023·Z1^2 -0.018·Z1Z2 -0.001·Z2^2 † their program
 100   (0, 0, 1)                      +1.833·X -0.022·Z1 -0.003·Z2 +0.016·X^2 -0.038·XZ1 +0.003·XZ2 +0.015·Z1^2 -0.007·Z1Z2 † their program
 100 (0, -1, -1)          -0.131·X -0.067·Z1 -0.011·Z2 +0.004·X^2 -0.014·XZ1 +0.012·XZ2 +0.013·Z1^2 +1.035·Z1Z2 -0.003·Z2^2 † their program
 500   (1, 0, 0)             +1.426·1 +0.300·X +1.050·Z1 +0.028·Z2 -0.032·X^2 +0.019·XZ1 -0.003·XZ2 -0.036·Z1^2 -0.003·Z1Z2 † their program
 500  (-1, 0, 0)    

### LaTeX tables

`writing/table_aic_lasso_comparison.tex`: for each active reaction and each $T$, the Bayesian posterior mean and the
mass-action LASSO estimate; the last rows summarise the inactive candidates.
`writing/table_aic_lasso_blackbox.tex`: the black-box polynomial of each channel at each $T$.

In [7]:
def tex_rxn(name):
    lhs, rhs = name.split("_to_")
    conv = lambda s: r"\varnothing" if s == "Empty" else s.replace("Z1", "Z_1").replace("Z2", "Z_2")
    return f"${conv(lhs)} \\to {conv(rhs)}$"

active_order = [reaction_names[i] for i in range(len(trueTheta)) if trueTheta[i] > 0]
L = [r"% Antithetic integral controller: Bayes vs. LASSO-MA versus T",
     r"% Generated by examples/example6_aic_lasso.ipynb -- do not edit by hand",
     r"\begin{table}[htbp]", r"\centering", r"\footnotesize", r"\setlength{\tabcolsep}{3pt}",
     r"\caption{Antithetic integral controller, prefixes of one trajectory up to time $T$. "
     r"For each active reaction: posterior mean (Bayes) and the estimate of LASSO-MA, the LASSO of "
     r"\cite{siam_Zhang2019} at $\lambda=" + f"{LAMBDA}" + r"$ restricted to the same mass-action candidate reactions. "
     r"Last rows: over the inactive candidate reactions of the observed channels, the smallest posterior "
     r"$\Pr(\text{off})$ (Bayes) and the largest $|\hat k|$ (LASSO-MA). "
     + (r"$^\ast$~LASSO-MA computed as the minimiser of the same objective with an independent solver (see text)."
        if any(T not in PROGRAM_T for T in T_LIST) else "") + r"}",
     r"\label{tab:aic_lasso_comparison}",
     r"\begin{tabular}{l r " + "r r " * len(T_LIST) + "}", r"\toprule",
     r" & & " + " & ".join(f"\\multicolumn{{2}}{{c}}{{$T={T}$}}" for T in T_LIST) + r" \\",
     "".join(f"\\cmidrule(lr){{{3+2*i}-{4+2*i}}}" for i in range(len(T_LIST))),
     r"Reaction & $k^*$ & " + " & ".join("Bayes & LASSO-MA" + ("" if T in PROGRAM_T else r"$^\ast$") for T in T_LIST) + r" \\", r"\midrule"]
for name in active_order:
    g = act[act["reaction"] == name].set_index("T")
    cells = [f"${g.loc[T, 'bayes_mean']:.3f}$ & ${g.loc[T, 'lasso_ma']:.3f}$" for T in T_LIST]
    L.append(f"{tex_rxn(name)} & {g['true'].iloc[0]:g} & " + " & ".join(cells) + r" \\")
L.append(r"\midrule")
mx = ina.groupby("T")["lasso_ma"].apply(lambda s: s.abs().max())
mn = ina.groupby("T")["pr_off"].min()
L.append(r"\multicolumn{2}{l}{inactive: min $\Pr(\text{off})$} & " + " & ".join(f"${mn[T]:.3f}$ & " for T in T_LIST) + r" \\")
L.append(r"\multicolumn{2}{l}{inactive: max $|\hat k|$} & " + " & ".join(f" & ${mx[T]:.3f}$" for T in T_LIST) + r" \\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
out1 = os.path.join(REPO, "writing", "table_aic_lasso_comparison.tex")
open(out1, "w").write("\n".join(L) + "\n")

def tex_poly(terms, bad):
    lab = lambda b: "" if len(b) == 0 else (
        ("X" if b[0] == 0 else f"Z_{b[0]}") if len(b) == 1 else
        ((("X" if b[0] == 0 else f"Z_{b[0]}") + "^2") if b[0] == b[1] else
         ("X" if b[0] == 0 else f"Z_{b[0]}") + ("X" if b[1] == 0 else f"Z_{b[1]}")))
    s = " ".join(f"{'-' if c < 0 else '+'} {abs(c):.3f}" + (f"\\,{lab(b)}" if lab(b) else "") for c, b in terms)
    s = s[2:] if s.startswith("+ ") else "-" + s[2:]
    dag = r"{}^{\,\dagger}" if bad else ""
    return f"${s}{dag}$"

L2 = [r"% Antithetic integral controller: LASSO-BB polynomials versus T",
      r"% Generated by examples/example6_aic_lasso.ipynb -- do not edit by hand",
      r"\begin{table}[htbp]", r"\centering", r"\small",
      r"\caption{Antithetic integral controller: propensity of each stoichiometric channel fitted by LASSO-BB, the LASSO of "
      r"\cite{siam_Zhang2019} at $\lambda=" + f"{LAMBDA}" + r"$ with all polynomials of order $\le 2$ in $(X, Z_1, Z_2)$, "
      r"terms with $|c|\ge 0.001$. "
      + (r"$^\ast$~minimiser of the same objective computed with an independent solver. "
         if any(T not in PROGRAM_T for T in T_LIST) else "") + r"$^\dagger$~contains a term not produced by any mass-action reaction with that "
      r"stoichiometric change, or a negative term. True propensities: $(+1,0,0)$: $2 + Z_1$; $(-1,0,0)$: $X$; "
      r"$(0,+1,0)$: $6$; $(0,0,+1)$: $2X$; $(0,-1,-1)$: $Z_1 Z_2$.}",
      r"\label{tab:aic_lasso_blackbox}",
      r"\begin{tabular}{l l p{0.72\textwidth}}", r"\toprule", r"$T$ & channel & LASSO-BB propensity \\", r"\midrule"]
for T in T_LIST:
    for j, ch in enumerate(CHANNELS):
        terms, bad, src = blackbox_poly(T, ch)
        star = r"$^\ast$" if src != "their program" else ""
        chs = "(" + ",".join(f"{v:+d}".replace("+0", "0") for v in ch) + ")"
        L2.append((f"{T}{star}" if j == 0 else "") + f" & ${chs}$ & {tex_poly(terms, bad)}" + r" \\")
    L2.append(r"\midrule")
L2[-1] = r"\bottomrule"
L2 += [r"\end{tabular}", r"\end{table}"]
out2 = os.path.join(REPO, "writing", "table_aic_lasso_blackbox.tex")
open(out2, "w").write("\n".join(L2) + "\n")
print("Saved:", out1, "\n      ", out2)
print("\n".join(L))


Saved: /sessions/rcw-01domso52nuiezllguqaql4l/mnt/BayesCRNInference/writing/table_aic_lasso_comparison.tex 
       /sessions/rcw-01domso52nuiezllguqaql4l/mnt/BayesCRNInference/writing/table_aic_lasso_blackbox.tex
% Antithetic integral controller: Bayes vs. LASSO-MA versus T
% Generated by examples/example6_aic_lasso.ipynb -- do not edit by hand
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\caption{Antithetic integral controller, prefixes of one trajectory up to time $T$. For each active reaction: posterior mean (Bayes) and the estimate of LASSO-MA, the LASSO of \cite{siam_Zhang2019} at $\lambda=0.01$ restricted to the same mass-action candidate reactions. Last rows: over the inactive candidate reactions of the observed channels, the smallest posterior $\Pr(\text{off})$ (Bayes) and the largest $|\hat k|$ (LASSO-MA). }
\label{tab:aic_lasso_comparison}
\begin{tabular}{l r r r r r r r r r }
\toprule
 & & \multicolumn{2}{c}{$T=100$} & \multicolumn{2}{c}{$T=500$} 